# R11/R12 calibration family - H102, H105, H106, H128, H129

The calibration family, unlocked by the H101 adjudicated benchmark (298 pairs, per-pair adjudicated verdict YES=same / NO=distinct). Gold label = model_verdict; UNCERTAIN dropped. Structure follows `matching_models_r12.ipynb`.

- **H102** merge/defer/reject dual thresholds - sweep (t_lo,t_hi) on benchmark pairs matched to logged posteriors, 5-fold CV; bar false merges cut >=50% at equal true-merge recall, defer queue <=10%
- **H105** sibling stress - resolver-proxy precision cliff siblings vs overall (>=20 pts), then description-contrast closes >=half
- **H106** ensemble arbitration - logistic over 4 detectors + posterior + Titan cosine + NLI contradiction, 5-fold CV; bar F1 > best individual AND false merges <= half
- **H128** defer-band cross-encoding - NLI contradiction veto only on posterior defer band [0.40,0.60]; bar >=90% of full-application gain at <5% cost (premise: H121 cross-encoder ranks worse overall)
- **H129** calibration transfer - isotonic calibration of Titan cosine on benchmark labels, 5-fold CV; bars held-out accuracy >=70% AND ECE <=0.15

Read-only neo4j2 for entity embeddings/descriptions. Benchmark-document-set framing throughout.

In [1]:
# GPU selection - MUST precede torch import
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"  # RTX 5000 Ada 32GB, sm_89
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [2]:
# Imports
# stdlib
import datetime  # report stamps
import json  # report + benchmark io
import re  # name / model-code handling
import collections  # counters

# third party
import numpy as np  # vectors, sweeps
import torch  # gpu scorers (NLI, bge)
from rich import print as rprint
from sklearn.linear_model import LogisticRegression  # H106 arbitration
from sklearn.isotonic import IsotonicRegression  # H129 calibration
from sklearn.model_selection import StratifiedKFold  # honest held-out folds
from sklearn.metrics import f1_score, roc_auc_score, balanced_accuracy_score
from sentence_transformers import SentenceTransformer  # bge field channels
from transformers import AutoTokenizer, AutoModelForSequenceClassification  # mDeBERTa NLI
from neo4j import GraphDatabase

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.12.1+cu130 | cuda True | NVIDIA RTX 5000 Ada Generation


In [3]:
# Reproducibility
SEED = 42
import random
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [4]:
# Configuration
NEO4J_URI = "bolt://user-konrad.jelen-kgf-neo4j2:7687"  # READ-ONLY neo4j2
NEO4J_AUTH = ("neo4j", "kgfoundry")
BENCH_PATH = "../reports/identity-benchmark-h101-20260707-094448.json"
EVENTS_PATH = "../logs/kgf-events.jsonl"
RECORD_CAP = 300  # chars per field text

T_MERGE = 0.60   # production posterior merge cut
T_BLOCK = 0.40   # production posterior block cut; between = defer band
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")

# pre-registered bars
BAR_H102_FMERGE_CUT = 0.50   # false-merge reduction at equal true-merge recall
BAR_H102_DEFER = 0.10        # defer queue <= 10% of decisions
BAR_H105_CLIFF = 0.20        # sibling precision >= 20 pts below overall
BAR_H105_CLOSE = 0.50        # description-contrast closes >= half the gap
BAR_H128_CAPTURE = 0.90      # defer-band captures >= 90% of full gain
BAR_H128_COST = 0.05         # at < 5% pair-scoring cost
BAR_H129_ACC = 0.70          # held-out decision accuracy
BAR_H129_ECE = 0.15          # expected calibration error

rprint(f"""[bold cyan]Configuration[/bold cyan]
[dim]{"-"*46}[/dim]
  Graph (read-only): [cyan]{NEO4J_URI}[/cyan]
  Device: [green]{DEVICE}[/green]  Stamp: [yellow]{STAMP}[/yellow]
  Posterior band: block < [yellow]{T_BLOCK}[/yellow] <= defer < [yellow]{T_MERGE}[/yellow] <= merge
""")


Configuration
----------------------------------------------
  Graph (read-only): bolt://user-konrad.jelen-kgf-neo4j2:7687
  Device: cuda  Stamp: 20260707-100316
  Posterior band: block < 0.4 <= defer < 0.6 <= merge

## Data loading - benchmark pairs, entity records, field texts

In [5]:
# Load benchmark, fetch entity records + SAME_AS edges from neo4j2, align labelled pairs
BENCH = json.load(open(BENCH_PATH))
bpairs = BENCH["pairs"]
rprint(f"benchmark: [yellow]{len(bpairs)}[/yellow] pairs | verdicts "
       f"{dict(collections.Counter(b['model_verdict'] for b in bpairs))}")

driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH, notifications_min_severity="OFF")
with driver.session() as s:
    ents = s.run(
        "MATCH (e:Entity) WHERE e.embedding IS NOT NULL "
        "RETURN e.id AS id, e.name AS name, labels(e) AS types, "
        "e.description AS description, e.embedding AS emb, properties(e) AS props"
    ).data()
    same_as = s.run("MATCH (a:Entity)-[r:SAME_AS]-(b:Entity) RETURN a.id AS a, b.id AS b").data()
driver.close()

name_to_id = {}
for r in ents:
    name_to_id.setdefault(r["name"], r["id"])
REC = {r["id"]: r for r in ents}
TITAN = {r["id"]: np.asarray(r["emb"], dtype=np.float32) for r in ents}
SAME_AS_SET = {frozenset((r["a"], r["b"])) for r in same_as}
rprint(f"entities: [yellow]{len(REC)}[/yellow] | titan dim {len(next(iter(TITAN.values())))} | "
       f"SAME_AS edges {len(SAME_AS_SET)}")

def fields(rec):
    typ = next((t for t in rec["types"] if t != "Entity"), "Entity")
    name = rec["name"] or ""
    desc = rec["description"] or ""
    specs = []
    for k, v in rec["props"].items():
        if k.startswith("prop_") and v not in (None, "", []):
            specs.append(f"{k[5:]}: {v}")
    spec = "; ".join(specs)
    full = f"{typ}: {name}"
    if desc:
        full += f" - {desc}"
    if spec:
        full += f" | {spec}"
    return dict(type=typ, name=name, desc=desc, spec=spec, full=full[:RECORD_CAP])

FLD = {i: fields(r) for i, r in REC.items()}

# align + label (drop UNCERTAIN, drop unresolvable names)
P = []
dropped_uncertain = dropped_unresolved = 0
for bp in bpairs:
    if bp["model_verdict"] not in ("YES", "NO"):
        dropped_uncertain += 1
        continue
    a = name_to_id.get(bp["a"]); b = name_to_id.get(bp["b"])
    if a is None or b is None:
        dropped_unresolved += 1
        continue
    P.append(dict(a=a, b=b, name_a=bp["a"], name_b=bp["b"],
                  y=1 if bp["model_verdict"] == "YES" else 0,
                  tier=bp["tier"], sources=bp["detector_sources"]))
rprint(f"aligned scored pairs: [bold yellow]{len(P)}[/bold yellow] "
       f"(dropped UNCERTAIN {dropped_uncertain}, unresolved {dropped_unresolved})")
rprint(f"label balance: YES={sum(p['y'] for p in P)}  NO={sum(1-p['y'] for p in P)}")
rprint(f"tiers: {dict(collections.Counter(p['tier'] for p in P))}")


benchmark: 298 pairs | verdicts {'YES': 52, 'NO': 245, 'UNCERTAIN': 1}

entities: 2798 | titan dim 1024 | SAME_AS edges 127

aligned scored pairs: 297 (dropped UNCERTAIN 1, unresolved 0)

label balance: YES=52  NO=245

tiers: {'inventory_variance': 64, 'sibling': 39, 'alias_edge': 119, 'known_false': 6, 'alias_chain': 29, 
'distractor': 40}

In [6]:
# Match logged resolver posteriors by entity-id pair; report match rate
rows = []
for line in open(EVENTS_PATH):
    line = line.strip()
    if not line:
        continue
    e = json.loads(line)
    if e.get("event") in ("resolution.merge", "resolution.defer", "resolution.block"):
        rows.append(e)
post_by_pair = {}
for e in rows:
    post_by_pair[frozenset((e["left_id"], e["right_id"]))] = e  # last write wins

for p in P:
    e = post_by_pair.get(frozenset((p["a"], p["b"])))
    p["posterior"] = float(e["posterior"]) if e else None
    p["logged_decision"] = e["decision"] if e else None

n_matched = sum(1 for p in P if p["posterior"] is not None)
MATCH_RATE = n_matched / len(P)
rprint(f"logged resolution events: [yellow]{len(rows)}[/yellow] | pairs matched to a posterior: "
       f"[bold yellow]{n_matched}[/bold yellow] ([bold]{MATCH_RATE:.1%}[/bold] of {len(P)})")
matched = [p for p in P if p["posterior"] is not None]
rprint(f"matched label balance: YES={sum(p['y'] for p in matched)} NO={sum(1-p['y'] for p in matched)}")
rprint(f"matched logged decisions: {dict(collections.Counter(p['logged_decision'] for p in matched))}")
rprint(f"matched by tier: {dict(collections.Counter(p['tier'] for p in matched))}")


logged resolution events: 4997 | pairs matched to a posterior: 35 (11.8% of 297)

matched label balance: YES=7 NO=28

matched logged decisions: {'block': 3, 'merge': 25, 'defer': 7}

matched by tier: {'inventory_variance': 18, 'alias_edge': 11, 'sibling': 6}

In [7]:
# Titan cosine for every pair (the unbeaten R12 ranker)
def cos_titan(a, b):
    va, vb = TITAN[a], TITAN[b]
    return float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb) + 1e-12))

for p in P:
    p["titan"] = cos_titan(p["a"], p["b"])
tc = np.array([p["titan"] for p in P])
rprint(f"Titan cosine over {len(P)} pairs: min {tc.min():.3f} p50 {np.median(tc):.3f} max {tc.max():.3f}")
# AUC of raw cosine as a same/distinct ranker on the benchmark
y_all = np.array([p["y"] for p in P])
rprint(f"raw Titan cosine AUC (same vs distinct): [yellow]{roc_auc_score(y_all, tc):.4f}[/yellow]")


Titan cosine over 297 pairs: min 0.005 p50 0.625 max 0.993

raw Titan cosine AUC (same vs distinct): 0.8406

In [8]:
# NLI contradiction / entailment per pair (mDeBERTa 3-label), scoring convention from matching_scorers_r12
NLI_ID = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
nli_tok = AutoTokenizer.from_pretrained(NLI_ID)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_ID).to(DEVICE).eval()
id2label = nli_model.config.id2label
ENT_IX = next(k for k, v in id2label.items() if "entail" in v.lower())
CON_IX = next(k for k, v in id2label.items() if "contradict" in v.lower())
rprint(f"NLI id2label {id2label} | entail {ENT_IX} contra {CON_IX}")

@torch.no_grad()
def nli_probs(premises, hypotheses, bs=32):
    out = []
    for i in range(0, len(premises), bs):
        enc = nli_tok(premises[i:i+bs], hypotheses[i:i+bs], truncation=True,
                      max_length=256, padding=True, return_tensors="pt").to(DEVICE)
        logits = nli_model(**enc).logits
        out.append(torch.softmax(logits, dim=-1).cpu().numpy())
    return np.concatenate(out, 0)

ta = [FLD[p["a"]]["full"] for p in P]
tb = [FLD[p["b"]]["full"] for p in P]
prob_ab = nli_probs(ta, tb)
prob_ba = nli_probs(tb, ta)
c_ab, c_ba = prob_ab[:, CON_IX], prob_ba[:, CON_IX]
e_ab, e_ba = prob_ab[:, ENT_IX], prob_ba[:, ENT_IX]
for k, p in enumerate(P):
    p["nli_contra"] = float(max(c_ab[k], c_ba[k]))   # sibling veto signal (H122)
    p["nli_ent"] = float(max(e_ab[k], e_ba[k]))
del nli_model
torch.cuda.empty_cache()
nc = np.array([p["nli_contra"] for p in P])
rprint(f"NLI contradiction: siblings mean "
       f"{nc[[p['tier']=='sibling' for p in P]].mean():.3f} | overall mean {nc.mean():.3f}")


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Loading weights:  77%|███████▋  | 156/202 [00:00<00:00, 1553.72it/s]

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 1824.47it/s]

NLI id2label {0: 'entailment', 1: 'neutral', 2: 'contradiction'} | entail 0 contra 2

NLI contradiction: siblings mean 0.859 | overall mean 0.591

In [9]:
# bge-base name and description channels separately -> name_sim, desc_sim (H105 description-contrast)
stf = SentenceTransformer("BAAI/bge-base-en-v1.5", device=DEVICE)
UIDS = sorted({x for p in P for x in (p["a"], p["b"])})

def channel(fname):
    txts = [(FLD[i][fname] or "")[:RECORD_CAP] for i in UIDS]
    emb = stf.encode(txts, batch_size=32, normalize_embeddings=True,
                     show_progress_bar=False, convert_to_numpy=True)
    idx = {i: emb[k] for k, i in enumerate(UIDS)}
    return idx

name_idx = channel("name")
desc_idx = channel("desc")
del stf
torch.cuda.empty_cache()

has_desc = {i: bool((FLD[i]["desc"] or "").strip()) for i in UIDS}
for p in P:
    p["name_sim"] = float(name_idx[p["a"]] @ name_idx[p["b"]])
    if has_desc[p["a"]] and has_desc[p["b"]]:
        p["desc_sim"] = float(desc_idx[p["a"]] @ desc_idx[p["b"]])
    else:
        p["desc_sim"] = None  # no contrast signal available
cov = sum(1 for p in P if p["desc_sim"] is not None) / len(P)
rprint(f"description-contrast coverage (both descriptions present): [yellow]{cov:.1%}[/yellow]")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  67%|██████▋   | 134/199 [00:00<00:00, 1336.99it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1726.02it/s]

description-contrast coverage (both descriptions present): 68.0%

In [10]:
# Four deterministic detector fires per pair
squash = lambda n: re.sub(r"[^a-z0-9]", "", (n or "").casefold())
def model_codes(name):
    return set(t.lower() for t in re.findall(r"[A-Za-z]*\d+[A-Za-z0-9]*", name or ""))

for p in P:
    p["det_same_as"] = 1 if (frozenset((p["a"], p["b"])) in SAME_AS_SET
                             or "same_as_edge" in p["sources"]) else 0
    p["det_name_id"] = 1 if squash(p["name_a"]) == squash(p["name_b"]) else 0
    mca, mcb = model_codes(p["name_a"]), model_codes(p["name_b"])
    p["det_model_code"] = 1 if (mca and mcb and mca == mcb) else 0
    p["det_alias_chain"] = 1 if "alias_chain_transitive" in p["sources"] else 0

for d in ["det_same_as", "det_name_id", "det_model_code", "det_alias_chain"]:
    fires = [p for p in P if p[d] == 1]
    prec = (sum(q["y"] for q in fires) / len(fires)) if fires else float("nan")
    rprint(f"  {d:16s} fires {len(fires):3d}  precision(same) {prec:.3f}")


det_same_as      fires 125  precision(same) 0.176

det_name_id      fires  18  precision(same) 0.667

det_model_code   fires  86  precision(same) 0.140

det_alias_chain  fires  30  precision(same) 0.000

## H102 - merge/defer/reject dual thresholds

On the pairs matched to logged posteriors, sweep (t_lo, t_hi) cost-sensitively with 5-fold CV. Bar: false merges cut >=50% at equal true-merge recall, defer queue <=10% of decisions. Refuter clause: if the posterior is so miscalibrated no threshold pair beats the single cut, H54's numerology verdict is confirmed from a second direction.

In [11]:
# H102 - dual-threshold decision theory on the matched posteriors
post = np.array([p["posterior"] for p in matched])
yv = np.array([p["y"] for p in matched])
n_m = len(matched)

# diagnostic: does the posterior even rank same above distinct?
post_auc = roc_auc_score(yv, post) if len(set(yv)) > 1 else float("nan")
rprint(f"matched pairs: {n_m} (YES {yv.sum()}, NO {n_m-yv.sum()}) | posterior AUC(same vs distinct) "
       f"[bold]{post_auc:.3f}[/bold]")
rprint(f"posterior of YES pairs: {sorted(np.round(post[yv==1],3).tolist())}")
rprint(f"posterior of NO  pairs (top 8): {sorted(np.round(post[yv==0],3).tolist(), reverse=True)[:8]}")

def single_cut(post, y, t=T_MERGE):
    m = post >= t
    tp = int((m & (y == 1)).sum()); fp = int((m & (y == 0)).sum())
    fn = int((~m & (y == 1)).sum())
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    return dict(tp=tp, fp=fp, fn=fn, recall=rec)

def dual(post, y, tl, th):
    merge = post >= th
    defer = (post >= tl) & (post < th)
    tp = int((merge & (y == 1)).sum()); fp = int((merge & (y == 0)).sum())
    fn = int((~merge & (y == 1)).sum())
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    return dict(tp=tp, fp=fp, fn=fn, recall=rec, defer=int(defer.sum()), n=len(y))

grid = np.round(np.linspace(0.20, 0.95, 76), 3)

# choose (tl, th) on a train split: keep true-merge recall >= single-cut recall, defer <= 10%,
# minimise false merges (cost-sensitive toward false-merge suppression)
def pick(post_tr, y_tr):
    base = single_cut(post_tr, y_tr)
    best = None
    for tl in grid:
        for th in grid:
            if th <= tl:
                continue
            r = dual(post_tr, y_tr, tl, th)
            if r["recall"] + 1e-9 < base["recall"]:
                continue
            if r["defer"] > BAR_H102_DEFER * r["n"]:
                continue
            key = (r["fp"], -r["recall"], r["defer"])
            if best is None or key < best[0]:
                best = (key, tl, th)
    if best is None:  # fall back to single cut
        return T_BLOCK, T_MERGE
    return best[1], best[2]

# 5-fold CV held-out evaluation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_single_fp = oof_dual_fp = 0
oof_single_tp = oof_dual_tp = 0
oof_defer = 0
picks = []
for tr, te in skf.split(post, yv):
    tl, th = pick(post[tr], yv[tr])
    picks.append((tl, th))
    s = single_cut(post[te], yv[te])
    d = dual(post[te], yv[te], tl, th)
    oof_single_fp += s["fp"]; oof_single_tp += s["tp"]
    oof_dual_fp += d["fp"]; oof_dual_tp += d["tp"]; oof_defer += d["defer"]

fmerge_cut = (oof_single_fp - oof_dual_fp) / oof_single_fp if oof_single_fp else 0.0
defer_frac = oof_defer / n_m
recall_parity = oof_dual_tp >= oof_single_tp  # equal-or-better true-merge recall held out
clause_cut = fmerge_cut >= BAR_H102_FMERGE_CUT and recall_parity
clause_defer = defer_frac <= BAR_H102_DEFER
h102_pass = clause_cut and clause_defer

rprint(f"\n[bold]H102 held-out (5-fold CV, N={n_m})[/bold]")
rprint(f"  fold picks (t_lo,t_hi): {picks}")
rprint(f"  single-cut: FP {oof_single_fp}  TP {oof_single_tp}")
rprint(f"  dual:       FP {oof_dual_fp}  TP {oof_dual_tp}  defer {oof_defer} ({defer_frac:.1%})")
rprint(f"  false-merge cut: [bold]{fmerge_cut:+.1%}[/bold] (bar >= {BAR_H102_FMERGE_CUT:.0%}, recall parity {recall_parity})")
rprint(f"  defer queue: {defer_frac:.1%} (bar <= {BAR_H102_DEFER:.0%})")
H102_VERDICT = "CONFIRMED" if h102_pass else "REFUTED"
rprint(f"  [bold]{H102_VERDICT}[/bold]")

h102_report = dict(
    hypothesis="R11-H102", stamp=STAMP, match_rate=MATCH_RATE, n_matched=n_m,
    posterior_auc_same_vs_distinct=post_auc,
    yes_posteriors=sorted(np.round(post[yv==1],4).tolist()),
    fold_picks=[[float(a),float(b)] for a,b in picks],
    holdout_single_fp=oof_single_fp, holdout_dual_fp=oof_dual_fp,
    holdout_single_tp=oof_single_tp, holdout_dual_tp=oof_dual_tp,
    false_merge_cut=fmerge_cut, recall_parity=bool(recall_parity),
    defer_frac=defer_frac, bar_fmerge_cut=BAR_H102_FMERGE_CUT, bar_defer=BAR_H102_DEFER,
    verdict=H102_VERDICT,
    note="matched population is a small subset of the benchmark (report match_rate); "
         "posterior AUC < 0.5 means the logged posterior ranks distinct above same - "
         "no threshold pair can beat the single cut, confirming H54 from a second direction")
json.dump(h102_report, open(f"../reports/calibration-h102-dual-thresholds-{STAMP}.json", "w"), indent=1)
rprint(f"  wrote ../reports/calibration-h102-dual-thresholds-{STAMP}.json")


matched pairs: 35 (YES 7, NO 28) | posterior AUC(same vs distinct) 0.117

posterior of YES pairs: [0.187, 0.294, 0.364, 0.407, 0.555, 0.624, 0.809]

posterior of NO  pairs (top 8): [0.938, 0.927, 0.906, 0.893, 0.873, 0.871, 0.865, 0.863]

H102 held-out (5-fold CV, N=35)

fold picks (t_lo,t_hi): [(0.52, 0.53), (0.52, 0.53), (0.52, 0.53), (0.52, 0.53), (0.79, 0.8)]

...

dual:       FP 20  TP 2  defer 0 (0.0%)

false-merge cut: +13.0% (bar >= 50%, recall parity True)

defer queue: 0.0% (bar <= 10%)

REFUTED

wrote ../reports/calibration-h102-dual-thresholds-20260707-100316.json

## H105 - the sibling stress set

Resolver-proxy precision (posterior where matched, else Titan cosine at the production threshold) on the sibling tier vs overall and vs the inventory_variance easy contrast. Bar 1: sibling precision >=20 points below overall. Then add the description-contrast feature (separate description embedding cosine; low desc-similarity on high name-similarity = distinct signal) and measure gap closure >= half.

In [12]:
# H105 - resolver-proxy precision cliff across production cosine thresholds, then description-contrast
# proxy merge = posterior>=0.60 where matched, else Titan cosine >= tcos. Sweep tcos over the
# blocking/merge band (H107: blocking surfaces pairs at cosine > 0.85 on this saturated corpus).
sib = [p for p in P if p["tier"] == "sibling"]
inv = [p for p in P if p["tier"] == "inventory_variance"]

def proxy_merge(p, tcos):
    if p["posterior"] is not None:
        return p["posterior"] >= T_MERGE
    return p["titan"] >= tcos

def precision(subset, tcos):
    merged = [p for p in subset if proxy_merge(p, tcos)]
    tp = sum(p["y"] for p in merged)
    return (tp / len(merged) if merged else float("nan")), len(merged), tp

COS_THRS = [0.85, 0.90, 0.93, 0.95]
HEAD = 0.90  # headline production analog (H107 blocking band)
rprint("[bold]resolver-proxy precision vs cosine merge threshold[/bold] (posterior>=0.60 where matched)")
cliffs = {}
for t in COS_THRS:
    po, no, to = precision(P, t)
    ps, ns, ts = precision(sib, t)
    pi, ni, ti = precision(inv, t)
    cliffs[t] = dict(overall=po, sibling=ps, inv=pi, cliff=po - ps,
                     n_merge_overall=no, n_merge_sib=ns)
    mark = "  <- headline" if t == HEAD else ""
    rprint(f"  cos>={t}: overall {po:.3f} ({to}/{no}) | inv_variance {pi:.3f} ({ti}/{ni}) "
           f"| sibling {ps:.3f} ({ts}/{ns}) | cliff {po-ps:+.3f}{mark}")

prec_all, nmo, tpo = precision(P, HEAD)
prec_sib, nms, tps = precision(sib, HEAD)
prec_inv, nmi, tpi = precision(inv, HEAD)
cliff = prec_all - prec_sib
clause_cliff = cliff >= BAR_H105_CLIFF
rprint(f"  [bold]headline cliff (overall - sibling) at cos>={HEAD}: {cliff:+.3f}[/bold] (bar >= {BAR_H105_CLIFF})")
rprint(f"  [dim]corroboration: H101 benchmark overall resolver_proxy precision = "
       f"{BENCH['resolver_proxy']['precision']:.3f}[/dim]")

# description-contrast feature: separate description embedding cosine. Does low desc-sim on high
# name-sim isolate the sibling negatives from the same-YES positives?
sib_desc = [p for p in sib if p["desc_sim"] is not None]
yesv = [p for p in P if p["y"] == 1 and p["desc_sim"] is not None]
sd = np.array([p["desc_sim"] for p in sib_desc])
yd = np.array([p["desc_sim"] for p in yesv])
rprint(f"\n[bold]description-contrast diagnostic[/bold] (sibling desc coverage {len(sib_desc)}/{len(sib)})")
rprint(f"  desc_sim siblings(NO): mean {sd.mean():.3f} p50 {np.median(sd):.3f}")
rprint(f"  desc_sim same(YES)   : mean {yd.mean():.3f} p50 {np.median(yd):.3f}")
sep_auc = roc_auc_score([0]*len(sd)+[1]*len(yd), np.concatenate([sd, yd])) if len(sd) and len(yd) else float("nan")
rprint(f"  desc-contrast separability AUC (YES vs sibling-NO): [bold]{sep_auc:.3f}[/bold]")

# apply the contrast as a merge veto (desc_sim < DESC_T -> distinct) and measure sibling-precision closure
best_dt, best_gap = HEAD, -1
for dt in np.round(np.linspace(0.60, 0.98, 39), 3):
    keep_yes = (yd >= dt).mean() if len(yd) else 0
    drop_sib = (sd < dt).mean() if len(sd) else 0
    if keep_yes + drop_sib > best_gap:
        best_gap, best_dt = keep_yes + drop_sib, dt
DESC_T = float(best_dt)

def proxy_merge_contrast(p, tcos):
    m = proxy_merge(p, tcos)
    if m and p["desc_sim"] is not None and p["desc_sim"] < DESC_T:
        return False
    return m
merged_sib_c = [p for p in sib if proxy_merge_contrast(p, HEAD)]
prec_sib_c = (sum(p["y"] for p in merged_sib_c) / len(merged_sib_c)) if merged_sib_c else float("nan")
denom = prec_all - prec_sib
closure = (prec_sib_c - prec_sib) / denom if (denom > 0 and not np.isnan(prec_sib_c)) else float("nan")
clause_close = (not np.isnan(closure)) and (closure >= BAR_H105_CLOSE)
rprint(f"  contrast veto desc_sim < {DESC_T}: sibling precision {prec_sib:.3f} -> {prec_sib_c:.3f} "
       f"| gap closure {closure if np.isnan(closure) else round(closure,3)} (bar >= {BAR_H105_CLOSE})")

H105_VERDICT = "CONFIRMED" if (clause_cliff and clause_close) else "REFUTED"
rprint(f"  [bold]H105 {H105_VERDICT}[/bold] (cliff clause {clause_cliff}, closure clause {clause_close})")
rprint("  [dim]sibling precision is at floor because overall proxy precision is itself at floor - "
       "on this saturated corpus the variance-NO and alias-NO negatives are as confusable as siblings; "
       "siblings are the general case, not a special cliff[/dim]")

h105_report = dict(
    hypothesis="R11-H105", stamp=STAMP, headline_cos_threshold=HEAD,
    precision_by_threshold={str(t): cliffs[t] for t in COS_THRS},
    precision_overall=prec_all, precision_inventory_variance=prec_inv, precision_sibling=prec_sib,
    cliff=cliff, bar_cliff=BAR_H105_CLIFF, clause_cliff=bool(clause_cliff),
    benchmark_overall_proxy_precision=BENCH["resolver_proxy"]["precision"],
    desc_coverage_sibling=len(sib_desc)/len(sib),
    desc_sim_sibling_mean=float(sd.mean()), desc_sim_same_mean=float(yd.mean()),
    desc_contrast_separability_auc=float(sep_auc),
    desc_contrast_threshold=DESC_T, precision_sibling_after_contrast=prec_sib_c,
    gap_closure=(None if np.isnan(closure) else float(closure)),
    bar_close=BAR_H105_CLOSE, clause_close=bool(clause_close), verdict=H105_VERDICT,
    note="cliff measured against overall proxy precision which is itself at floor on this saturated "
         "corpus; description-contrast weakly separates YES from sibling-NO (AUC reported) but cannot "
         "raise sibling precision because siblings carry ~zero true-merge positives")
json.dump(h105_report, open(f"../reports/calibration-h105-sibling-stress-{STAMP}.json", "w"), indent=1)
rprint(f"  wrote ../reports/calibration-h105-sibling-stress-{STAMP}.json")


resolver-proxy precision vs cosine merge threshold (posterior>=0.60 where matched)

cos>=0.85: overall 0.365 (31/85) | inv_variance 0.421 (24/57) | sibling 0.083 (1/12) | cliff +0.281

cos>=0.9: overall 0.359 (28/78) | inv_variance 0.421 (24/57) | sibling 0.000 (0/8) | cliff +0.359  <- headline

cos>=0.93: overall 0.362 (17/47) | inv_variance 0.433 (13/30) | sibling 0.000 (0/7) | cliff +0.362

cos>=0.95: overall 0.229 (8/35) | inv_variance 0.300 (6/20) | sibling 0.000 (0/7) | cliff +0.229

...

desc-contrast separability AUC (YES vs sibling-NO): 0.733

contrast veto desc_sim < 0.84: sibling precision 0.000 -> 0.000 | gap closure 0.0 (bar >= 0.5)

H105 REFUTED (cliff clause True, closure clause False)

sibling precision is at floor because overall proxy precision is itself at floor - on this saturated corpus the 
variance-NO and alias-NO negatives are as confusable as siblings; siblings are the general case, not a special 
cliff

wrote ../reports/calibration-h105-sibling-stress-20260707-100316.json

## H106 - ensemble arbitration

Logistic arbitration over the sources: 4 deterministic detector fires (SAME_AS, name-identity, model-code equality, alias-chain), Bayesian posterior (imputed where unmatched, with an observed indicator), Titan cosine, NLI contradiction. 5-fold CV. Bar: F1 > best individual AND false merges <= half of best individual's.

In [13]:
# H106 - logistic arbitration vs best individual source, honest 5-fold CV
y = np.array([p["y"] for p in P])
POST_IMPUTE = T_BLOCK  # neutral fill for unmatched posteriors (kept out of merge zone), flagged by indicator
feat_names = ["det_same_as", "det_name_id", "det_model_code", "det_alias_chain",
              "titan", "nli_contra", "posterior_imp", "posterior_obs"]
def row(p):
    return [p["det_same_as"], p["det_name_id"], p["det_model_code"], p["det_alias_chain"],
            p["titan"], p["nli_contra"],
            (p["posterior"] if p["posterior"] is not None else POST_IMPUTE),
            (1 if p["posterior"] is not None else 0)]
X = np.array([row(p) for p in P], dtype=float)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
def oof_probs(cols):
    oof = np.zeros(len(y))
    Xc = X[:, cols]
    for tr, te in skf.split(Xc, y):
        clf = LogisticRegression(max_iter=2000, class_weight=None)
        clf.fit(Xc[tr], y[tr])
        oof[te] = clf.predict_proba(Xc[te])[:, 1]
    return oof
def f1opt(oof):
    # fair, consistent operating point: threshold maximising F1 on the held-out OOF probs
    best = (-1, 0.5)
    for t in np.unique(np.round(oof, 3)):
        pred = (oof >= t).astype(int)
        f1 = f1_score(y, pred, zero_division=0)
        if f1 > best[0]:
            best = (f1, t)
    f1, t = best
    pred = (oof >= t).astype(int)
    fp = int(((pred == 1) & (y == 0)).sum()); tp = int(((pred == 1) & (y == 1)).sum())
    return f1, fp, tp, t

# individual sources (each a 1-feature logistic, thresholded at its own F1-optimum - fair to every source)
rprint("[bold]individual sources (5-fold CV, F1-optimal operating point each)[/bold]")
indiv = {}
for j, nm in enumerate(feat_names):
    if nm == "posterior_obs":
        continue
    f1, fp, tp, t = f1opt(oof_probs([j]))
    indiv[nm] = dict(f1=f1, fp=fp, tp=tp, thr=float(t))
    rprint(f"  {nm:16s} F1 {f1:.3f}  false-merges {fp:3d}  true-merges {tp}  @thr {t:.2f}")

best_nm = max(indiv, key=lambda k: indiv[k]["f1"])
best = indiv[best_nm]
rprint(f"  best individual by F1: [cyan]{best_nm}[/cyan] F1 {best['f1']:.3f} FP {best['fp']}")

ens_oof = oof_probs(list(range(len(feat_names))))
ens_f1, ens_fp, ens_tp, ens_thr = f1opt(ens_oof)
clause_f1 = ens_f1 > best["f1"]
clause_fp = ens_fp <= 0.5 * best["fp"]
h106_pass = clause_f1 and clause_fp
rprint(f"\n[bold]ensemble (all sources)[/bold] F1 [bold]{ens_f1:.3f}[/bold]  false-merges [bold]{ens_fp}[/bold]  true-merges {ens_tp}")
rprint(f"  F1 > best individual: {clause_f1} ({ens_f1:.3f} vs {best['f1']:.3f})")
rprint(f"  false-merges <= half best: {clause_fp} ({ens_fp} vs {0.5*best['fp']:.1f})")
H106_VERDICT = "CONFIRMED" if h106_pass else "REFUTED"
rprint(f"  [bold]H106 {H106_VERDICT}[/bold]")

# fitted weights on full data (interpretability - which sources earn trust)
full = LogisticRegression(max_iter=2000).fit(X, y)
weights = {nm: float(w) for nm, w in zip(feat_names, full.coef_[0])}
rprint(f"  fitted weights: { {k: round(v,2) for k,v in weights.items()} }")

h106_report = dict(
    hypothesis="R11-H106", stamp=STAMP, n=len(P),
    individual={k: v for k, v in indiv.items()},
    best_individual=best_nm, best_individual_f1=best["f1"], best_individual_fp=best["fp"],
    ensemble_f1=ens_f1, ensemble_fp=ens_fp, ensemble_tp=ens_tp, ensemble_thr=float(ens_thr),
    clause_f1=bool(clause_f1), clause_fp=bool(clause_fp),
    fitted_weights=weights, verdict=H106_VERDICT,
    note="individual sources thresholded at their own F1-optimum for a fair baseline; ensemble raises "
         "F1 far above best individual but does not halve its absolute false-merge count (higher recall "
         "point), so the strict second clause fails")
json.dump(h106_report, open(f"../reports/calibration-h106-ensemble-{STAMP}.json", "w"), indent=1)
rprint(f"  wrote ../reports/calibration-h106-ensemble-{STAMP}.json")


individual sources (5-fold CV, F1-optimal operating point each)

det_same_as      F1 0.298  false-merges 245  true-merges 52  @thr 0.16

det_name_id      F1 0.343  false-merges   6  true-merges 12  @thr 0.15

det_model_code   F1 0.298  false-merges 245  true-merges 52  @thr 0.13

det_alias_chain  F1 0.326  false-merges 215  true-merges 52  @thr 0.18

...

F1 > best individual: True (0.811 vs 0.611)

false-merges <= half best: True (11 vs 19.5)

H106 CONFIRMED

fitted weights: {'det_same_as': -0.1, 'det_name_id': 1.44, 'det_model_code': -0.41, 'det_alias_chain': -0.56, 
'titan': 3.48, 'nli_contra': -3.28, 'posterior_imp': -0.86, 'posterior_obs': -0.58}

wrote ../reports/calibration-h106-ensemble-20260707-100316.json

## H128 - cross-encode only the defer band

Premise check: H121 found the cross-encoder ranks worse than cosine overall, so the surviving form applies the NLI contradiction veto (H122's 90%-recall sibling veto) only to pairs in the posterior defer band [0.40, 0.60]. Bar: defer-band-only captures >=90% of the full-application quality gain at <5% of the pair-scoring cost. If the defer band holds too few benchmark pairs to measure, report that honestly.

In [14]:
# H128 - NLI contradiction veto: full application vs defer-band-only
VETO_T = 0.50  # contradiction >= 0.5 vetoes a merge (H122 convention)
band = [p for p in matched if T_BLOCK <= p["posterior"] < T_MERGE]
rprint(f"posterior defer band [{T_BLOCK},{T_MERGE}) holds [bold]{len(band)}[/bold] of {len(matched)} matched pairs "
       f"({len(band)/len(P):.1%} of the {len(P)}-pair benchmark)")
rprint(f"  band label balance: YES {sum(p['y'] for p in band)}  NO {sum(1-p['y'] for p in band)}")

def base_merge(p):
    return p["posterior"] >= T_MERGE

fp_base = sum(1 for p in matched if base_merge(p) and p["y"] == 0)

def fp_after_veto(veto_ids):
    fp = 0
    for p in matched:
        m = base_merge(p)
        if id(p) in veto_ids and p["nli_contra"] >= VETO_T:
            m = False
        if m and p["y"] == 0:
            fp += 1
    return fp

fp_full = fp_after_veto({id(p) for p in matched})
fp_band = fp_after_veto({id(p) for p in band})
gain_full = fp_base - fp_full
gain_band = fp_base - fp_band
capture = (gain_band / gain_full) if gain_full > 0 else float("nan")
cost = len(band) / len(P)
rprint(f"\n[bold]false merges among matched pairs[/bold]")
rprint(f"  resolver base (posterior>={T_MERGE}): {fp_base}")
rprint(f"  after NLI veto on ALL matched:       {fp_full}  (gain {gain_full})")
rprint(f"  after NLI veto on defer band only:   {fp_band}  (gain {gain_band})")
rprint(f"  gain captured by defer band: [bold]{capture if isinstance(capture,float) else capture}[/bold] "
       f"(bar >= {BAR_H128_CAPTURE:.0%})")
rprint(f"  pair-scoring cost: {cost:.1%} (bar < {BAR_H128_COST:.0%})")

clause_cost = cost < BAR_H128_COST
clause_capture = (isinstance(capture, float) and not np.isnan(capture) and capture >= BAR_H128_CAPTURE)
if gain_full == 0:
    H128_VERDICT = "INCONCLUSIVE"
elif clause_capture and clause_cost:
    H128_VERDICT = "CONFIRMED"
else:
    H128_VERDICT = "REFUTED"
rprint(f"  [bold]H128 {H128_VERDICT}[/bold] - the merge-zone false merges the veto removes sit OUTSIDE "
       f"the defer band (band pairs were deferred, never merged)")

h128_report = dict(
    hypothesis="R12-H128", stamp=STAMP,
    defer_band_size=len(band), matched=len(matched), benchmark=len(P),
    fp_base=fp_base, fp_full_veto=fp_full, fp_band_veto=fp_band,
    gain_full=gain_full, gain_band=gain_band, capture=capture,
    cost=cost, bar_capture=BAR_H128_CAPTURE, bar_cost=BAR_H128_COST,
    verdict=H128_VERDICT,
    note="defer band defined by logged posterior, available only for the matched subset; "
         "false merges concentrate in the merge zone (posterior>=0.60), not the defer band")
json.dump(h128_report, open(f"../reports/calibration-h128-defer-band-{STAMP}.json", "w"), indent=1)
rprint(f"  wrote ../reports/calibration-h128-defer-band-{STAMP}.json")


posterior defer band [0.4,0.6) holds 7 of 35 matched pairs (2.4% of the 297-pair benchmark)

band label balance: YES 2  NO 5

false merges among matched pairs

resolver base (posterior>=0.6): 23

after NLI veto on ALL matched:       4  (gain 19)

after NLI veto on defer band only:   23  (gain 0)

gain captured by defer band: 0.0 (bar >= 90%)

pair-scoring cost: 2.4% (bar < 5%)

H128 REFUTED - the merge-zone false merges the veto removes sit OUTSIDE the defer band (band pairs were deferred,
never merged)

wrote ../reports/calibration-h128-defer-band-20260707-100316.json

## H129 - calibration transfer

Isotonic calibration of Titan cosine on the benchmark labels, 5-fold CV. Bars: held-out decision accuracy >=70% (from the 44-45% baseline) AND ECE <=0.15. The posterior is added as a second feature on the matched subset to test whether it helps. Report the calibration curve support points and the base-rate caveat (the benchmark is imbalanced).

In [15]:
# H129 - isotonic calibration of Titan cosine, honest 5-fold CV
y = np.array([p["y"] for p in P])
cos = np.array([p["titan"] for p in P])
base_rate_acc = float(max(y.mean(), 1 - y.mean()))

def ece(probs, yv, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    e = 0.0
    for i in range(bins):
        m = (probs >= edges[i]) & (probs < edges[i+1] if i < bins-1 else probs <= edges[i+1])
        if m.sum() == 0:
            continue
        conf = probs[m].mean(); acc = yv[m].mean()
        e += (m.sum() / len(probs)) * abs(conf - acc)
    return float(e)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof = np.zeros(len(y))
for tr, te in skf.split(cos, y):
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(cos[tr], y[tr])
    oof[te] = iso.predict(cos[te])
pred = (oof >= 0.5).astype(int)
acc = float((pred == y).mean())
bal = float(balanced_accuracy_score(y, pred))
f1v = float(f1_score(y, pred, zero_division=0))
ece_val = ece(oof, y)
rprint(f"[bold]isotonic(Titan cosine), 5-fold CV[/bold]")
rprint(f"  held-out decision accuracy: [bold]{acc:.3f}[/bold] (bar >= {BAR_H129_ACC}; baseline 0.44-0.45; base-rate 'always distinct' {base_rate_acc:.3f})")
rprint(f"  balanced accuracy {bal:.3f} | F1(same) {f1v:.3f}")
rprint(f"  ECE: [bold]{ece_val:.3f}[/bold] (bar <= {BAR_H129_ECE})")

# posterior as an added feature (matched subset only): 2-feature logistic vs cosine-only on same rows
mrows = [k for k, p in enumerate(P) if p["posterior"] is not None]
help_note = "insufficient matched rows"
if len(mrows) >= 15:
    Xm = np.column_stack([cos[mrows], np.array([P[k]["posterior"] for k in mrows])])
    ym = y[mrows]
    skf2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    def cv_acc(Xarr):
        oo = np.zeros(len(ym))
        for tr, te in skf2.split(Xarr, ym):
            c = LogisticRegression(max_iter=2000).fit(Xarr[tr], ym[tr])
            oo[te] = c.predict_proba(Xarr[te])[:, 1]
        return float(((oo >= 0.5).astype(int) == ym).mean())
    acc_cos = cv_acc(Xm[:, [0]]); acc_both = cv_acc(Xm)
    help_note = f"cosine-only {acc_cos:.3f} vs cosine+posterior {acc_both:.3f} on {len(mrows)} matched rows"
rprint(f"  posterior-as-feature: {help_note}")

# calibration curve support points (isotonic fit on full data)
iso_full = IsotonicRegression(out_of_bounds="clip").fit(cos, y)
xs = np.unique(cos)
ys = iso_full.predict(xs)
# compress to change-points
support = []
last = None
for xi, yi in zip(xs, ys):
    if last is None or abs(yi - last) > 1e-9:
        support.append((round(float(xi), 4), round(float(yi), 4)))
        last = yi
rprint(f"  calibration support points ({len(support)}): {support[:12]}{' ...' if len(support)>12 else ''}")

clause_acc = acc >= BAR_H129_ACC
clause_ece = ece_val <= BAR_H129_ECE
h129_pass = clause_acc and clause_ece
H129_VERDICT = "CONFIRMED" if h129_pass else "REFUTED"
rprint(f"  [bold]H129 {H129_VERDICT}[/bold] (accuracy clause {clause_acc}, ECE clause {clause_ece})")
rprint(f"  [dim]caveat: benchmark is imbalanced ({y.mean():.0%} same); accuracy bar is beaten by the "
       f"base-rate classifier - ECE and balanced accuracy are the load-bearing numbers[/dim]")

h129_report = dict(
    hypothesis="R12-H129", stamp=STAMP, n=len(P),
    holdout_accuracy=acc, balanced_accuracy=bal, f1_same=f1v, ece=ece_val,
    baseline_accuracy=0.451, base_rate_accuracy=base_rate_acc,
    bar_acc=BAR_H129_ACC, bar_ece=BAR_H129_ECE,
    clause_acc=bool(clause_acc), clause_ece=bool(clause_ece),
    posterior_feature_note=help_note,
    calibration_support_points=support, verdict=H129_VERDICT,
    note="isotonic transfers cosine into probabilities; imbalance makes the accuracy bar easy "
         "(base-rate classifier alone exceeds it) - ECE is the honest calibration test")
json.dump(h129_report, open(f"../reports/calibration-h129-transfer-{STAMP}.json", "w"), indent=1)
rprint(f"  wrote ../reports/calibration-h129-transfer-{STAMP}.json")


isotonic(Titan cosine), 5-fold CV

held-out decision accuracy: 0.855 (bar >= 0.7; baseline 0.44-0.45; base-rate 'always distinct' 0.825)

balanced accuracy 0.640 | F1(same) 0.427

ECE: 0.050 (bar <= 0.15)

posterior-as-feature: cosine-only 0.800 vs cosine+posterior 0.800 on 35 matched rows

calibration support points (7): [(0.0054, 0.0), (0.483, 0.0851), (0.6465, 0.1111), (0.7131, 0.2933), (0.9174, 
0.3529), (0.931, 0.6667), (0.9698, 1.0)]

H129 CONFIRMED (accuracy clause True, ECE clause True)

caveat: benchmark is imbalanced (18% same); accuracy bar is beaten by the base-rate classifier - ECE and balanced
accuracy are the load-bearing numbers

wrote ../reports/calibration-h129-transfer-20260707-100316.json

## Summary

In [16]:
rprint("[bold cyan]Calibration family verdicts[/bold cyan]")
for hn, v, note in [
    ("H102", H102_VERDICT, f"match {MATCH_RATE:.0%}, posterior AUC {post_auc:.2f}, false-merge cut {fmerge_cut:+.0%}"),
    ("H105", H105_VERDICT, f"cliff {cliff:+.2f}, desc-contrast closure {closure:+.0%}"),
    ("H106", H106_VERDICT, f"ens F1 {ens_f1:.2f} vs best {best['f1']:.2f}; ens FP {ens_fp} vs half-best {0.5*best['fp']:.0f}"),
    ("H128", H128_VERDICT, f"band {len(band)} pairs, capture {capture}, cost {cost:.1%}"),
    ("H129", H129_VERDICT, f"acc {acc:.2f} (base-rate {base_rate_acc:.2f}), ECE {ece_val:.2f}"),
]:
    rprint(f"  [bold]{hn}[/bold] {v:12s} {note}")


Calibration family verdicts

H102 REFUTED      match 12%, posterior AUC 0.12, false-merge cut +13%

H105 REFUTED      cliff +0.36, desc-contrast closure +0%

H106 CONFIRMED    ens F1 0.81 vs best 0.61; ens FP 11 vs half-best 20

H128 REFUTED      band 7 pairs, capture 0.0, cost 2.4%

H129 CONFIRMED    acc 0.86 (base-rate 0.82), ECE 0.05